In [ ]:
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


def plot_seed_comparisons(
    seeds: list[int],
    save: bool,
    methods: list[str] | str,
    heatmap_path_templates: list[str] | str,
    vmax: list[int] | int,
    prefix: str = "",
    show_colorbars: bool = True,
):
    """
    Generates one figure containing all seed comparisons.

    Each seed is shown as one row and each method as one column.
    When `show_colorbars` is True, every row has its own vertical colorbar.
    """
    if isinstance(vmax, int):
        vmax = [vmax]

    if isinstance(methods, str):
        methods = [methods]

    if isinstance(heatmap_path_templates, str):
        heatmap_path_templates = [heatmap_path_templates]

    if len(methods) != len(heatmap_path_templates):
        raise ValueError(
            "`methods` and `heatmap_path_templates` must have the same length."
        )

    if not seeds:
        raise ValueError("At least one seed must be provided.")

    num_rows = len(seeds)
    num_methods = len(methods)

    # Add one GridSpec column only when colorbars are enabled.
    num_grid_cols = num_methods + int(show_colorbars)

    figure_width = 5 * num_methods
    if show_colorbars:
        figure_width += 0.9

    fig = plt.figure(
        figsize=(figure_width, 5 * num_rows),
    )

    width_ratios = [1] * num_methods
    if show_colorbars:
        width_ratios.append(0.08)

    gs = GridSpec(
        num_rows,
        num_grid_cols,
        figure=fig,
        width_ratios=width_ratios,
        wspace=0.05,
        hspace=0.1,
    )

    heatmap_axes = [
        [
            fig.add_subplot(gs[row, col])
            for col in range(num_methods)
        ]
        for row in range(num_rows)
    ]

    if show_colorbars:
        colorbar_axes = [
            fig.add_subplot(gs[row, -1])
            for row in range(num_rows)
        ]
    else:
        colorbar_axes = []

    for row_idx, seed in enumerate(seeds):
        row_rightmost_im = None

        for col_idx, (method, heatmap_path_template, cur_vmax) in enumerate(
            zip(methods, heatmap_path_templates, vmax)
        ):
            ax_heat = heatmap_axes[row_idx][col_idx]

            method_label = (
                method.split("/")[-1]
                .replace("_", " ")
                .replace("-", " ")
                .title()
            )
            method_label = method_label.replace("<s>", str(seed))

            heatmap_path = heatmap_path_template.format(seed=seed)
            is_empty = False

            try:
                df_heat = pd.read_csv(heatmap_path)
                df_heat.columns = ["alpha", "beta", "value"]

                pivot_df = df_heat.pivot(
                    index="alpha",
                    columns="beta",
                    values="value",
                )
            except Exception:
                pivot_df = pd.DataFrame(
                    np.nan,
                    index=np.linspace(-0.5, 1.5, 200),
                    columns=np.linspace(-0.5, 1.5, 200),
                )
                is_empty = True

            im = ax_heat.imshow(
                pivot_df,
                vmin=0,
                vmax=cur_vmax,
            )

            if col_idx == num_methods - 1:
                row_rightmost_im = im

            if is_empty:
                ax_heat.text(
                    0.5,
                    0.5,
                    "No Data Available",
                    ha="center",
                    va="center",
                    color="gray",
                    fontsize=12,
                    transform=ax_heat.transAxes,
                )

            label_data = [
                (50, 50, r"$\theta_{1}$"),
                (150, 50, r"$\theta_{2}$"),
                (150, 150, r"$\theta_{3}$"),
            ]

            x_coords = [point[0] for point in label_data]
            y_coords = [point[1] for point in label_data]

            x_coords.append(label_data[0][0])
            y_coords.append(label_data[0][1])

            ax_heat.plot(
                x_coords,
                y_coords,
                color="red",
                linestyle="--",
                linewidth=2,
            )

            ax_heat.set_xticks([])
            ax_heat.set_yticks([])

            for x, y, label in label_data:
                ax_heat.text(
                    x,
                    y,
                    label,
                    color="white",
                    fontsize=15,
                    ha="center",
                    va="center",
                    bbox={
                        "boxstyle": "round,pad=0.3",
                        "facecolor": "black",
                        "edgecolor": "none",
                        "alpha": 0.6,
                    },
                )

            if row_idx == 0:
                ax_heat.set_title(
                    method_label,
                    pad=10,
                    fontsize=35,
                )

            if col_idx == 0:
                ax_heat.set_ylabel(
                    f"Seed {seed}",
                    fontsize=35,
                    labelpad=10,
                )

        # ======================================================
        # Manual colorbar for this row
        # ======================================================
        if show_colorbars and row_rightmost_im is not None:
            cax = colorbar_axes[row_idx]

            gradient = np.linspace(0, vmax[0], 512).reshape(-1, 1)

            cax.imshow(
                gradient,
                cmap=row_rightmost_im.get_cmap(),
                norm=row_rightmost_im.norm,
                aspect="auto",
                origin="lower",
            )

            cax.set_xticks([])

            cax.set_yticks(
                np.linspace(0, 511, 5),
                labels=[0, 25, 50, 75, 100],
            )

            cax.yaxis.tick_right()
            cax.tick_params(
                axis="y",
                labelsize=25,
                length=0,
                pad=8,
            )

            for spine in cax.spines.values():
                spine.set_visible(False)

    if save:
        fig.savefig(
            f"../output/{prefix}_heatmaps.svg",
            format="svg",
            bbox_inches="tight",
        )

    plt.show()

In [ ]:
if __name__ == "__main__":
    plot_seed_comparisons(
        seeds=list(range(50)),
        methods=[
            "SAC",
            "DDPG",
            "DQN",
            "SACD",
        ],
        heatmap_path_templates=[
            "../output/interpolations/sac_no_encode_eauto/data_{seed}.csv",
            "../output/interpolations/ddpg_no_encode/data_{seed}.csv",
            "../output/interpolations/dqn_no_encode/data_{seed}.csv",
            "../output/interpolations/sacd_no_encode_eauto/data_{seed}.csv",
        ],
        save=False,
        prefix='sac',
        vmax=[1000, 1000, 500, 500],
        show_colorbars=False,
    )
